# Neural Embedding Search

We build a semantic search engine using neural embeddings.

Unlike TF-IDF, this method captures meaning, not just keyword overlap.


In [28]:
import sys
print(sys.executable)

/Users/PREACHERBOYKEYS/TextMiningAndNaturalLanguageProcessingHomeAssignment/nlp-env/bin/python


In [29]:
import pandas as pd

df = pd.read_csv("../Data/final_tweets.csv")

print(df.shape)
df.head()

(59184, 4)


,tweet_id,text,clean_text,company
0,277379,Another great set of @Delta flights thank to #...,another great set of flights thank to tmobilew...,delta
1,2488664,@AppleSupport why is my iPhone automatically g...,why is my iphone automatically going on mute,applesupport
2,2383194,@Uber_Support Since yesterday I haven't receiv...,since yesterday i havent received any update a...,uber_support
3,1610886,"@AmazonHelp There is no email from 2 days, jus...",there is no email from days just asked to wait...,amazonhelp
4,2763258,"@SouthwestAir thanks for responding, will call...",thanks for responding will call as soon as i g...,southwestair


In [30]:
import sys
!{sys.executable} -m pip install sentence-transformers


In [31]:
from sentence_transformers import SentenceTransformer

In [32]:
!pip install ipywidgets

In [33]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully


In [34]:
search_df = df[["tweet_id", "clean_text", "company"]].copy()

search_df.head()

,tweet_id,clean_text,company
0,277379,another great set of flights thank to tmobilew...,delta
1,2488664,why is my iphone automatically going on mute,applesupport
2,2383194,since yesterday i havent received any update a...,uber_support
3,1610886,there is no email from days just asked to wait...,amazonhelp
4,2763258,thanks for responding will call as soon as i g...,southwestair


In [35]:
texts = search_df["clean_text"].tolist()

embeddings = model.encode(texts, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/1850 [00:00<?, ?it/s]

Embeddings shape: (59184, 384)


In [36]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query, top_k=5):
    # embed query
    query_vec = model.encode([query])
    
    # compute similarity
    scores = cosine_similarity(query_vec, embeddings).flatten()
    
    # get top results
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = search_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    
    return results

In [37]:
semantic_search("flight delayed customer service", top_k=5)

,tweet_id,clean_text,company,score
2472,1799160,flight was already delayed on top of it poor c...,americanair,0.856027
41166,2721339,flight delayed and theres a hr wait on custome...,delta,0.789575
24784,366638,flight delayed hours but no texts no emails no...,delta,0.776327
46536,2258648,been there done that with customer service thr...,americanair,0.773329
36349,2098673,my flight has been delayed hours and no custom...,americanair,0.769100


The neural embedding model captures semantic meaning rather than relying on exact keyword overlap. It retrieves tweets related to flight delays and customer service complaints even when different wording is used, demonstrating improved contextual understanding compared to TF-IDF.